# 08 — Deduplicar y fusionar publicaciones

Este notebook deduplica publicaciones **solo dentro de la misma `Fuente_origen`**.

Principios:
- `Fuente_origen` es la frontera de deduplicación.
- `Base_origen` no impide fusionar.
- DOI distintos no se fusionan automáticamente.
- Se conserva una fila por publicación retenida + investigador UNAM.
- `SubArea` permanece vacía.
- Se generan únicamente los cuatro archivos solicitados.


In [1]:
import os
import re
import html
import unicodedata
from itertools import combinations

import pandas as pd

entrada = "../04_Limpieza/03_limpieza_bibliografica/autores_unam_limpios.csv"
carpeta_salida = "../04_Limpieza/04_deduplicacion"

os.makedirs(carpeta_salida, exist_ok=True)

columnas = [
    "Base_origen", "Fuente_origen", "indice", "Titulo", "Año",
    "Autor_norm", "Afiliacion1", "Afiliacion2", "ISBN", "ISSN",
    "Doi", "URL", "Area", "SubArea", "Keywords", "Abstract"
]

df = pd.read_csv(
    entrada,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig"
)
df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].copy()
df = df[columnas].copy()

print("Filas totales:", len(df))
print("Autores únicos:", df["Autor_norm"].nunique())
print("SubArea con valor:", df["SubArea"].fillna("").astype(str).str.strip().ne("").sum())

Filas totales: 4266
Autores únicos: 581
SubArea con valor: 0


In [2]:
def texto(x):
    return "" if pd.isna(x) else str(x).strip()

def normalizar_doi(x):
    s = texto(x).lower()
    s = re.sub(r"^https?://(?:dx\.)?doi\.org/", "", s)
    s = re.sub(r"^doi\s*:\s*", "", s)
    return s.strip().rstrip(".,;")

def normalizar_titulo(x):
    s = html.unescape(texto(x))
    s = unicodedata.normalize("NFKC", s).lower()

    for guion in ["‐", "‑", "‒", "–", "—", "―"]:
        s = s.replace(guion, "-")

    s = "".join(c if c.isalnum() or c.isspace() else " " for c in s)
    return re.sub(r"\s+", " ", s).strip()

def normalizar_simple(x):
    s = html.unescape(texto(x))
    s = unicodedata.normalize("NFKC", s).lower()
    return re.sub(r"\s+", " ", s).strip()

def normalizar_url(x):
    return normalizar_simple(x).rstrip("/")

def normalizar_id(x):
    return re.sub(r"[^0-9a-zx]", "", normalizar_simple(x))

def normalizar_keywords(x):
    s = normalizar_simple(x)
    if not s:
        return ""

    partes = [re.sub(r"\s+", " ", p).strip() for p in re.split(r"[;,]", s)]
    return ";".join(sorted({p for p in partes if p}))

def primero_no_vacio(serie):
    for x in serie:
        if texto(x):
            return x
    return pd.NA

In [3]:
claves_publicacion = ["Base_origen", "Fuente_origen", "indice"]

publicaciones = (
    df.groupby(claves_publicacion, dropna=False)
    .agg(
        Titulo=("Titulo", primero_no_vacio),
        Año=("Año", primero_no_vacio),
        Doi=("Doi", primero_no_vacio),
        ISBN=("ISBN", primero_no_vacio),
        ISSN=("ISSN", primero_no_vacio),
        URL=("URL", primero_no_vacio),
        Area=("Area", primero_no_vacio),
        Keywords=("Keywords", primero_no_vacio),
        Abstract=("Abstract", primero_no_vacio),
        autores=("Autor_norm", lambda s: tuple(sorted({texto(x) for x in s if texto(x)}))),
        n_filas=("Autor_norm", "size")
    )
    .reset_index()
)

publicaciones["_doi"] = publicaciones["Doi"].map(normalizar_doi)
publicaciones["_titulo"] = publicaciones["Titulo"].map(normalizar_titulo)
publicaciones["_año"] = pd.to_numeric(publicaciones["Año"], errors="coerce").astype("Int64")
publicaciones["_ISBN"] = publicaciones["ISBN"].map(normalizar_id)
publicaciones["_ISSN"] = publicaciones["ISSN"].map(normalizar_id)
publicaciones["_URL"] = publicaciones["URL"].map(normalizar_url)
publicaciones["_Abstract"] = publicaciones["Abstract"].map(normalizar_simple)
publicaciones["_Keywords"] = publicaciones["Keywords"].map(normalizar_keywords)

print("Publicaciones originales:", len(publicaciones))
print("\nPublicaciones por Fuente_origen:")
print(publicaciones["Fuente_origen"].value_counts().to_string())

doi_repetidos = publicaciones[publicaciones["_doi"] != ""].groupby(["Fuente_origen", "_doi"]).size()
titulo_repetidos = publicaciones[publicaciones["_titulo"] != ""].groupby(["Fuente_origen", "_titulo"]).size()

doi_entre_fuentes = (
    publicaciones[publicaciones["_doi"] != ""]
    .groupby("_doi")["Fuente_origen"].nunique()
)

titulo_entre_fuentes = (
    publicaciones[publicaciones["_titulo"] != ""]
    .groupby("_titulo")["Fuente_origen"].nunique()
)

print("\nGrupos DOI repetido dentro de una fuente:", int((doi_repetidos > 1).sum()))
print("Grupos título repetido dentro de una fuente:", int((titulo_repetidos > 1).sum()))
print("DOI en más de una Fuente_origen, solo reporte:", int((doi_entre_fuentes > 1).sum()))
print("Títulos en más de una Fuente_origen, solo reporte:", int((titulo_entre_fuentes > 1).sum()))

Publicaciones originales: 2109

Publicaciones por Fuente_origen:
Fuente_origen
Scopus           1261
IEEE              383
EV                251
WoS                83
ACM                66
ScienceDirect      51
ProQuest           14

Grupos DOI repetido dentro de una fuente: 567
Grupos título repetido dentro de una fuente: 584
DOI en más de una Fuente_origen, solo reporte: 150
Títulos en más de una Fuente_origen, solo reporte: 154


In [4]:
def evidencia_adicional(p, q):
    evidencia = []

    if set(p["autores"]) & set(q["autores"]):
        evidencia.append("AUTOR")

    for campo in ["_ISBN", "_ISSN", "_URL", "_Abstract", "_Keywords"]:
        if p[campo] and q[campo] and p[campo] == q[campo]:
            evidencia.append(campo[1:].upper())

    return evidencia

# En este archivo los candidatos relevantes aparecen por DOI exacto
# o título normalizado exacto. No fue necesario fuzzy matching.
pares = set()

for (_, _), grupo in publicaciones[publicaciones["_doi"] != ""].groupby(["Fuente_origen", "_doi"]):
    for i, j in combinations(grupo.index.tolist(), 2):
        pares.add((min(i, j), max(i, j)))

for (_, _), grupo in publicaciones[publicaciones["_titulo"] != ""].groupby(["Fuente_origen", "_titulo"]):
    for i, j in combinations(grupo.index.tolist(), 2):
        pares.add((min(i, j), max(i, j)))

filas_candidatos = []

for i, j in sorted(pares):
    p = publicaciones.loc[i]
    q = publicaciones.loc[j]

    if p["Fuente_origen"] != q["Fuente_origen"]:
        raise ValueError("ERROR: candidato entre Fuente_origen diferentes")

    doi_a = p["_doi"]
    doi_b = q["_doi"]

    titulo_exacto = bool(p["_titulo"]) and p["_titulo"] == q["_titulo"]

    año_a = p["_año"]
    año_b = q["_año"]
    ambos_años = pd.notna(año_a) and pd.notna(año_b)
    mismo_año = ambos_años and int(año_a) == int(año_b)
    año_conflictivo = ambos_años and int(año_a) != int(año_b)

    evidencia = evidencia_adicional(p, q)

    clasificacion = "POSIBLE_DUPLICADO"
    motivo = ""
    prioridad = 9

    if doi_a and doi_b and doi_a == doi_b:
        if titulo_exacto and not año_conflictivo:
            clasificacion = "DUPLICADO_SEGURO"
            motivo = "DOI exacto + título normalizado exacto"
            prioridad = 1
        else:
            motivo = "DOI exacto, pero título/año requiere revisión"

    elif doi_a and doi_b and doi_a != doi_b:
        if titulo_exacto and año_conflictivo:
            clasificacion = "NO_DUPLICADO"
            motivo = "Título exacto, pero DOI y año diferentes"
        elif titulo_exacto and mismo_año:
            motivo = "Título y año iguales, pero DOI diferentes"
        else:
            clasificacion = "NO_DUPLICADO"
            motivo = "DOI diferentes y evidencia insuficiente"

    else:
        if titulo_exacto and mismo_año and evidencia:
            clasificacion = "DUPLICADO_SEGURO"
            motivo = "Título exacto + mismo año + evidencia: " + ", ".join(evidencia)
            prioridad = 2
        elif titulo_exacto and mismo_año:
            motivo = "Título exacto + mismo año, sin evidencia adicional"
        elif titulo_exacto and año_conflictivo:
            clasificacion = "NO_DUPLICADO"
            motivo = "Título exacto, pero año diferente y sin DOI común"
        else:
            motivo = "Coincidencia parcial que requiere revisión"

    filas_candidatos.append({
        "i": i,
        "j": j,
        "Clasificacion": clasificacion,
        "Motivo": motivo,
        "Prioridad": prioridad,
        "Similitud_titulo": 100.0 if titulo_exacto else 0.0,
        "Evidencia_adicional": "; ".join(evidencia)
    })

clasificados = pd.DataFrame(filas_candidatos)

print("\nClasificación de pares:")
print(clasificados["Clasificacion"].value_counts().to_string())


Clasificación de pares:
Clasificacion
DUPLICADO_SEGURO     3323
POSIBLE_DUPLICADO      17
NO_DUPLICADO            1


In [5]:
n = len(publicaciones)

padre = list(range(n))
miembros = {i: {i} for i in range(n)}
dois_grupo = {
    i: ({publicaciones.at[i, "_doi"]} if publicaciones.at[i, "_doi"] else set())
    for i in range(n)
}
años_grupo = {
    i: ({int(publicaciones.at[i, "_año"])} if pd.notna(publicaciones.at[i, "_año"]) else set())
    for i in range(n)
}

def buscar(x):
    while padre[x] != x:
        padre[x] = padre[padre[x]]
        x = padre[x]
    return x

def unir(a, b):
    raiz_a = buscar(a)
    raiz_b = buscar(b)

    if raiz_a == raiz_b:
        return True

    if len(dois_grupo[raiz_a] | dois_grupo[raiz_b]) > 1:
        return False

    if len(años_grupo[raiz_a] | años_grupo[raiz_b]) > 1:
        return False

    fuentes = {
        publicaciones.at[k, "Fuente_origen"]
        for k in (miembros[raiz_a] | miembros[raiz_b])
    }

    if len(fuentes) != 1:
        return False

    if len(miembros[raiz_a]) < len(miembros[raiz_b]):
        raiz_a, raiz_b = raiz_b, raiz_a

    padre[raiz_b] = raiz_a
    miembros[raiz_a] |= miembros[raiz_b]
    dois_grupo[raiz_a] |= dois_grupo[raiz_b]
    años_grupo[raiz_a] |= años_grupo[raiz_b]

    del miembros[raiz_b]
    del dois_grupo[raiz_b]
    del años_grupo[raiz_b]

    return True

seguros = clasificados[
    clasificados["Clasificacion"] == "DUPLICADO_SEGURO"
].sort_values(["Prioridad", "i", "j"])

for indice_fila, fila in seguros.iterrows():
    if not unir(int(fila["i"]), int(fila["j"])):
        clasificados.loc[indice_fila, "Clasificacion"] = "POSIBLE_DUPLICADO"
        clasificados.loc[indice_fila, "Motivo"] += " | no fusionado por conflicto de DOI/año en el grupo"

publicaciones["_raiz"] = [buscar(i) for i in range(n)]

tamaños = publicaciones["_raiz"].value_counts()
raices_fusionadas = set(tamaños[tamaños > 1].index)

for raiz in raices_fusionadas:
    if publicaciones.loc[publicaciones["_raiz"] == raiz, "Fuente_origen"].nunique() != 1:
        raise ValueError("ERROR: grupo fusionado con más de una Fuente_origen")

print("Grupos DUPLICADO_SEGURO:", len(raices_fusionadas))
print("Publicaciones después:", publicaciones["_raiz"].nunique())

Grupos DUPLICADO_SEGURO: 586
Publicaciones después: 648


In [6]:
campos_completitud = [
    "Titulo", "Año", "Doi", "ISBN", "ISSN",
    "URL", "Area", "Keywords", "Abstract"
]

def puntaje_representante(p):
    completitud = sum(texto(p[c]) != "" for c in campos_completitud)

    doi_original = texto(p["Doi"]).lower()
    doi_limpio = int(
        bool(doi_original)
        and doi_original.startswith("10.")
        and doi_original == normalizar_doi(doi_original)
    )

    titulo_original = texto(p["Titulo"])
    titulo_limpio = int(titulo_original == html.unescape(titulo_original).strip())

    return (
        completitud,
        doi_limpio + titulo_limpio,
        len(titulo_original),
        len(texto(p["Abstract"]))
    )

representante = {}

for raiz, grupo in publicaciones.groupby("_raiz"):
    representante[raiz] = max(
        grupo.index.tolist(),
        key=lambda i: puntaje_representante(publicaciones.loc[i])
    )

raices_ordenadas = sorted(
    raices_fusionadas,
    key=lambda r: (
        str(publicaciones.loc[representante[r], "Fuente_origen"]),
        str(publicaciones.loc[representante[r], "Base_origen"]),
        str(publicaciones.loc[representante[r], "indice"])
    )
)

id_grupo = {r: f"G{n:04d}" for n, r in enumerate(raices_ordenadas, 1)}

motivos_por_grupo = {r: set() for r in raices_fusionadas}

for fila in clasificados[
    clasificados["Clasificacion"] == "DUPLICADO_SEGURO"
].itertuples():
    raiz = buscar(int(fila.i))
    if raiz in motivos_por_grupo and buscar(int(fila.j)) == raiz:
        motivos_por_grupo[raiz].add(fila.Motivo)

motivos_por_grupo = {
    r: " | ".join(sorted(motivos))
    for r, motivos in motivos_por_grupo.items()
}

In [7]:
publicaciones["_pub_id"] = publicaciones.index

filas = df.merge(
    publicaciones[claves_publicacion + ["_pub_id", "_raiz"]],
    on=claves_publicacion,
    how="left",
    validate="many_to_one"
)

def elegir_doi(grupo, rep_i):
    valor = publicaciones.loc[rep_i, "Doi"]

    if texto(valor):
        return valor

    valores = [x for x in grupo["Doi"] if texto(x)]
    if not valores:
        return pd.NA

    return max(
        valores,
        key=lambda x: (texto(x).lower().startswith("10."), -len(texto(x)))
    )

def elegir_url(grupo, rep_i):
    valor = publicaciones.loc[rep_i, "URL"]

    if texto(valor).lower().startswith(("http://", "https://")):
        return valor

    valores = [x for x in grupo["URL"] if texto(x)]
    validas = [x for x in valores if texto(x).lower().startswith(("http://", "https://"))]

    if validas:
        return validas[0]

    return valores[0] if valores else pd.NA

def unir_keywords(valores):
    resultado = []
    vistos = set()

    for x in valores:
        if not texto(x):
            continue

        for parte in re.split(r"[;,]", texto(x)):
            parte = re.sub(r"\s+", " ", parte).strip()
            clave = normalizar_simple(parte)

            if parte and clave not in vistos:
                vistos.add(clave)
                resultado.append(parte)

    return "; ".join(resultado) if resultado else pd.NA

def mas_largo(valores):
    valores = [x for x in valores if texto(x)]
    return max(valores, key=lambda x: len(texto(x))) if valores else pd.NA

In [8]:
salida_filas = []
conflictos_revision = []

for raiz, grupo_publicaciones in publicaciones.groupby("_raiz"):
    rep_i = representante[raiz]
    rep = publicaciones.loc[rep_i]

    ids_publicacion = set(grupo_publicaciones.index)
    grupo_filas = filas[filas["_pub_id"].isin(ids_publicacion)].copy()

    titulo = rep["Titulo"] if texto(rep["Titulo"]) else mas_largo(grupo_publicaciones["Titulo"])
    año = rep["Año"] if texto(rep["Año"]) else primero_no_vacio(grupo_publicaciones["Año"])
    doi = elegir_doi(grupo_publicaciones, rep_i)
    isbn = rep["ISBN"] if texto(rep["ISBN"]) else primero_no_vacio(grupo_publicaciones["ISBN"])
    issn = rep["ISSN"] if texto(rep["ISSN"]) else primero_no_vacio(grupo_publicaciones["ISSN"])
    url = elegir_url(grupo_publicaciones, rep_i)
    area = rep["Area"]
    keywords = unir_keywords(grupo_publicaciones["Keywords"])
    abstract = mas_largo(grupo_publicaciones["Abstract"])

    areas = [texto(x) for x in grupo_publicaciones["Area"] if texto(x)]

    if len(set(areas)) > 1 and len(grupo_publicaciones) > 1:
        conflictos_revision.append({
            "Tipo_caso": "CONFLICTO_AREA",
            "ID_grupo": id_grupo.get(raiz, ""),
            "Fuente_origen": rep["Fuente_origen"],
            "Base_origen_A": rep["Base_origen"],
            "indice_A": rep["indice"],
            "Base_origen_B": "",
            "indice_B": "",
            "Titulo_A": titulo,
            "Titulo_B": "",
            "Año_A": año,
            "Año_B": "",
            "Doi_A": doi,
            "Doi_B": "",
            "Similitud_titulo": "",
            "Motivo": "Áreas distintas dentro de un grupo seguro: " + " | ".join(sorted(set(areas)))
        })

    for autor, grupo_autor in grupo_filas.groupby("Autor_norm", dropna=False, sort=False):
        afiliaciones = []

        for _, fila_autor in grupo_autor.iterrows():
            for campo in ["Afiliacion1", "Afiliacion2"]:
                valor = texto(fila_autor[campo])
                if valor and valor not in afiliaciones:
                    afiliaciones.append(valor)

        filas_rep = grupo_autor[grupo_autor["_pub_id"] == rep_i]
        ordenadas = []

        for bloque in [filas_rep, grupo_autor]:
            for _, fila_autor in bloque.iterrows():
                for campo in ["Afiliacion1", "Afiliacion2"]:
                    valor = texto(fila_autor[campo])
                    if valor and valor not in ordenadas:
                        ordenadas.append(valor)

        if len(afiliaciones) > 2 and len(grupo_publicaciones) > 1:
            conflictos_revision.append({
                "Tipo_caso": "CONFLICTO_AFILIACION",
                "ID_grupo": id_grupo.get(raiz, ""),
                "Fuente_origen": rep["Fuente_origen"],
                "Base_origen_A": rep["Base_origen"],
                "indice_A": rep["indice"],
                "Base_origen_B": "",
                "indice_B": "",
                "Titulo_A": titulo,
                "Titulo_B": "",
                "Año_A": año,
                "Año_B": "",
                "Doi_A": doi,
                "Doi_B": "",
                "Similitud_titulo": "",
                "Motivo": f"{autor}: más de dos afiliaciones distintas: " + " | ".join(afiliaciones)
            })

        salida_filas.append({
            "Base_origen": rep["Base_origen"],
            "Fuente_origen": rep["Fuente_origen"],
            "indice": rep["indice"],
            "Titulo": titulo,
            "Año": año,
            "Autor_norm": autor,
            "Afiliacion1": ordenadas[0] if ordenadas else pd.NA,
            "Afiliacion2": ordenadas[1] if len(ordenadas) > 1 else pd.NA,
            "ISBN": isbn,
            "ISSN": issn,
            "Doi": doi,
            "URL": url,
            "Area": area,
            "SubArea": "",
            "Keywords": keywords,
            "Abstract": abstract
        })

autores_unam_deduplicados = pd.DataFrame(salida_filas, columns=columnas)
autores_unam_deduplicados["Año"] = pd.to_numeric(
    autores_unam_deduplicados["Año"], errors="coerce"
).astype("Int64")

In [9]:
candidatos_salida = []

for n, fila in enumerate(clasificados.itertuples(), 1):
    p = publicaciones.loc[int(fila.i)]
    q = publicaciones.loc[int(fila.j)]

    candidatos_salida.append({
        "ID_candidato": f"C{n:05d}",
        "Fuente_origen": p["Fuente_origen"],
        "Base_origen_A": p["Base_origen"],
        "indice_A": p["indice"],
        "Titulo_A": p["Titulo"],
        "Año_A": p["Año"],
        "Doi_A": p["Doi"],
        "Base_origen_B": q["Base_origen"],
        "indice_B": q["indice"],
        "Titulo_B": q["Titulo"],
        "Año_B": q["Año"],
        "Doi_B": q["Doi"],
        "Similitud_titulo": fila.Similitud_titulo,
        "Evidencia_adicional": fila.Evidencia_adicional,
        "Clasificacion": fila.Clasificacion,
        "Motivo": fila.Motivo
    })

candidatos_duplicados = pd.DataFrame(candidatos_salida)

posibles_por_raiz = {}

for fila in clasificados[
    clasificados["Clasificacion"] == "POSIBLE_DUPLICADO"
].itertuples():
    raiz_a = buscar(int(fila.i))
    raiz_b = buscar(int(fila.j))

    if raiz_a == raiz_b:
        continue

    clave = tuple(sorted((raiz_a, raiz_b)))
    posibles_por_raiz.setdefault(clave, fila)

casos_revision = []

for (raiz_a, raiz_b), fila in posibles_por_raiz.items():
    rep_a = publicaciones.loc[representante[raiz_a]]
    rep_b = publicaciones.loc[representante[raiz_b]]

    casos_revision.append({
        "Tipo_caso": "POSIBLE_DUPLICADO",
        "ID_grupo": "",
        "Fuente_origen": rep_a["Fuente_origen"],
        "Base_origen_A": rep_a["Base_origen"],
        "indice_A": rep_a["indice"],
        "Base_origen_B": rep_b["Base_origen"],
        "indice_B": rep_b["indice"],
        "Titulo_A": rep_a["Titulo"],
        "Titulo_B": rep_b["Titulo"],
        "Año_A": rep_a["Año"],
        "Año_B": rep_b["Año"],
        "Doi_A": rep_a["Doi"],
        "Doi_B": rep_b["Doi"],
        "Similitud_titulo": fila.Similitud_titulo,
        "Motivo": fila.Motivo
    })

casos_revision.extend(conflictos_revision)

columnas_revision = [
    "Tipo_caso", "ID_grupo", "Fuente_origen",
    "Base_origen_A", "indice_A", "Base_origen_B", "indice_B",
    "Titulo_A", "Titulo_B", "Año_A", "Año_B",
    "Doi_A", "Doi_B", "Similitud_titulo", "Motivo"
]

casos_revision_deduplicacion = pd.DataFrame(
    casos_revision,
    columns=columnas_revision
)

In [10]:
trazabilidad = []

for raiz in raices_fusionadas:
    rep_i = representante[raiz]
    rep = publicaciones.loc[rep_i]

    ids_publicacion = set(
        publicaciones.loc[publicaciones["_raiz"] == raiz].index
    )

    filas_grupo = filas[filas["_pub_id"].isin(ids_publicacion)]

    for _, fila_original in filas_grupo.iterrows():
        trazabilidad.append({
            "ID_grupo": id_grupo[raiz],
            "Fuente_origen": fila_original["Fuente_origen"],
            "Base_origen_original": fila_original["Base_origen"],
            "indice_original": fila_original["indice"],
            "Titulo_original": fila_original["Titulo"],
            "Doi_original": fila_original["Doi"],
            "Autor_norm": fila_original["Autor_norm"],
            "Base_origen_representante": rep["Base_origen"],
            "indice_representante": rep["indice"],
            "Motivo_fusion": motivos_por_grupo[raiz]
        })

trazabilidad_fusion = pd.DataFrame(trazabilidad)

In [11]:
# VALIDACIONES OBLIGATORIAS

assert autores_unam_deduplicados.columns.tolist() == columnas
assert autores_unam_deduplicados["Autor_norm"].fillna("").astype(str).str.strip().ne("").all()

autores_antes = set(df["Autor_norm"].fillna("").astype(str).str.strip())
autores_despues = set(autores_unam_deduplicados["Autor_norm"].fillna("").astype(str).str.strip())
assert autores_antes == autores_despues

for raiz in raices_fusionadas:
    # validación explícita solicitada
    assert publicaciones.loc[publicaciones["_raiz"] == raiz, "Fuente_origen"].nunique() == 1

    dois = {
        normalizar_doi(x)
        for x in publicaciones.loc[publicaciones["_raiz"] == raiz, "Doi"]
        if normalizar_doi(x)
    }
    assert len(dois) <= 1

# Ningún caso dudoso se fusionó
for fila in clasificados[
    clasificados["Clasificacion"] == "POSIBLE_DUPLICADO"
].itertuples():
    assert buscar(int(fila.i)) != buscar(int(fila.j))

# No queda ningún par seguro entre dos grupos distintos
seguros_restantes = 0
for fila in clasificados[
    clasificados["Clasificacion"] == "DUPLICADO_SEGURO"
].itertuples():
    if buscar(int(fila.i)) != buscar(int(fila.j)):
        seguros_restantes += 1

assert seguros_restantes == 0

assert autores_unam_deduplicados["SubArea"].fillna("").astype(str).str.strip().eq("").all()

filas_en_fusiones = filas[filas["_raiz"].isin(raices_fusionadas)]
trazabilidad_completa = len(trazabilidad_fusion) == len(filas_en_fusiones)
assert trazabilidad_completa

grupos_multibase = sum(
    publicaciones.loc[publicaciones["_raiz"] == raiz, "Base_origen"].nunique() > 1
    for raiz in raices_fusionadas
)

print("Validaciones correctas.")
print("Grupos seguros con más de una Base_origen:", grupos_multibase)

Validaciones correctas.
Grupos seguros con más de una Base_origen: 314


In [12]:
ruta_principal = f"{carpeta_salida}/autores_unam_deduplicados.csv"
ruta_candidatos = f"{carpeta_salida}/candidatos_duplicados.csv"
ruta_revision = f"{carpeta_salida}/casos_revision_deduplicacion.csv"
ruta_trazabilidad = f"{carpeta_salida}/trazabilidad_fusion.csv"

autores_unam_deduplicados.to_csv(ruta_principal, index=False, encoding="utf-8-sig")
candidatos_duplicados.to_csv(ruta_candidatos, index=False, encoding="utf-8-sig")
casos_revision_deduplicacion.to_csv(ruta_revision, index=False, encoding="utf-8-sig")
trazabilidad_fusion.to_csv(ruta_trazabilidad, index=False, encoding="utf-8-sig")

por_fuente_antes = publicaciones["Fuente_origen"].value_counts().sort_index()
por_fuente_despues = (
    autores_unam_deduplicados[["Base_origen", "Fuente_origen", "indice"]]
    .drop_duplicates()["Fuente_origen"]
    .value_counts()
    .sort_index()
)

print("\nRESUMEN FINAL")
print("Filas antes:", len(df))
print("Filas después:", len(autores_unam_deduplicados))
print("Publicaciones antes:", len(publicaciones))
print("Publicaciones después:", publicaciones["_raiz"].nunique())

print("\nPublicaciones por Fuente_origen antes:")
print(por_fuente_antes.to_string())

print("\nPublicaciones por Fuente_origen después:")
print(por_fuente_despues.to_string())

print("\nGrupos DUPLICADO_SEGURO:", len(raices_fusionadas))
print("Grupos POSIBLE_DUPLICADO:", len(posibles_por_raiz))
print("Filas fusionadas:", len(df) - len(autores_unam_deduplicados))
print("Autores únicos antes:", df["Autor_norm"].nunique())
print("Autores únicos después:", autores_unam_deduplicados["Autor_norm"].nunique())
print("Fusiones entre diferentes Fuente_origen: 0")
print("Casos pendientes:", len(casos_revision_deduplicacion))
print("Trazabilidad completa:", trazabilidad_completa)


RESUMEN FINAL
Filas antes: 4266
Filas después: 1331
Publicaciones antes: 2109
Publicaciones después: 648

Publicaciones por Fuente_origen antes:
Fuente_origen
ACM                66
EV                251
IEEE              383
ProQuest           14
ScienceDirect      51
Scopus           1261
WoS                83

Publicaciones por Fuente_origen después:
Fuente_origen
ACM               18
EV               115
IEEE             100
ProQuest           9
ScienceDirect     25
Scopus           360
WoS               21

Grupos DUPLICADO_SEGURO: 586
Grupos POSIBLE_DUPLICADO: 3
Filas fusionadas: 2935
Autores únicos antes: 581
Autores únicos después: 581
Fusiones entre diferentes Fuente_origen: 0
Casos pendientes: 17
Trazabilidad completa: True


In [13]:
# Comprobación final de archivos generados
archivos_generados = [
    ruta_principal,
    ruta_candidatos,
    ruta_revision,
    ruta_trazabilidad,
]

for archivo in archivos_generados:
    assert os.path.exists(archivo), f"No se generó: {archivo}"

print("\nArchivos guardados correctamente en:")
print(carpeta_salida)
for archivo in archivos_generados:
    print("-", archivo)



Archivos guardados correctamente en:
../04_Limpieza/04_deduplicacion
- ../04_Limpieza/04_deduplicacion/autores_unam_deduplicados.csv
- ../04_Limpieza/04_deduplicacion/candidatos_duplicados.csv
- ../04_Limpieza/04_deduplicacion/casos_revision_deduplicacion.csv
- ../04_Limpieza/04_deduplicacion/trazabilidad_fusion.csv
